# Data Understanding

*Generated by DoML — Do Machine Learning*

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

# Project root — REPR-02 (non-negotiable per CLAUDE.md)
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))

with open(PROJECT_ROOT / '.planning' / 'config.json') as f:
    config = json.load(f)

# Column names for headerless CSVs — keyed by file stem or file name
# e.g. {"iris": ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]}
column_names = config.get('column_names', {})

print(f"Project root:  {PROJECT_ROOT}")
print(f"Problem type:  {config.get('problem_type', 'unknown')}")
print(f"Time factor:   {config.get('time_factor', False)}")
print(f"Language:      {config.get('language', 'python')}")
print(f"Dataset path:  {config.get('dataset', {}).get('path', 'data/raw/')}")
if column_names:
    print(f"Column names:  {list(column_names.keys())}")

## 1. Data Profiling

<!-- EDA-02 -->
Zero-copy scan of `data/raw/` — schema, row counts, null counts, numeric distributions, and categorical unique counts via SQL.

*No pandas load required for this section.*

In [ ]:
import duckdb
import pandas as pd

data_dir = PROJECT_ROOT / 'data' / 'raw'
SUPPORTED = {'.csv', '.parquet', '.xlsx'}
files = sorted([f for f in data_dir.glob('**/*') if f.is_file() and f.suffix.lower() in SUPPORTED])

if not files:
    print(f"No supported files found in {data_dir}")
    print("Supported formats: CSV (.csv), Parquet (.parquet), Excel (.xlsx)")
else:
    print(f"Found {len(files)} file(s) in data/raw/:")
    for path in files:
        print(f"  {path.name}")

def get_read_fn(path):
    """Return a DuckDB read-function fragment for the given file.
    Applies user-supplied column names from config.column_names for headerless CSVs."""
    p = str(path)
    ext = path.suffix.lower()
    if ext == '.csv':
        user_names = column_names.get(path.stem) or column_names.get(path.name)
        if user_names:
            names_sql = '[' + ', '.join(f"'{n}'" for n in user_names) + ']'
            return f"read_csv('{p}', header=false, names={names_sql})"
        return f"read_csv_auto('{p}')"
    elif ext == '.parquet':
        return f"read_parquet('{p}')"
    elif ext == '.xlsx':
        return f"read_xlsx('{p}')"

for path in files:
    con = duckdb.connect()
    read_fn = get_read_fn(path)

    schema_df = con.execute(f"DESCRIBE SELECT * FROM {read_fn}").df()
    row_count = con.execute(f"SELECT COUNT(*) FROM {read_fn}").fetchone()[0]
    cols = schema_df['column_name'].tolist()

    null_exprs = ', '.join(
        f"SUM(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) AS \"{c}\""
        for c in cols
    )
    null_df = con.execute(f"SELECT {null_exprs} FROM {read_fn}").df().T.rename(columns={0: 'null_count'})
    null_df['null_pct'] = (null_df['null_count'] / row_count * 100).round(1)
    con.close()

    display(Markdown(f"### {path.name}"))
    print(f"Shape: {row_count:,} rows \u00d7 {len(cols)} columns")
    display(Markdown("**Schema (DuckDB DESCRIBE):**"))
    display(schema_df[['column_name', 'column_type']].rename(columns={'column_name': 'Column', 'column_type': 'Type'}))
    has_nulls = null_df[null_df['null_count'] > 0]
    if len(has_nulls) > 0:
        display(Markdown("**Null counts (columns with nulls only):**"))
        display(has_nulls)
    else:
        print("\u2713 No null values found")
    print()

In [ ]:
NUMERIC_TYPES = {'INTEGER', 'BIGINT', 'DOUBLE', 'FLOAT', 'HUGEINT', 'SMALLINT', 'TINYINT', 'DECIMAL', 'NUMERIC', 'REAL'}

for path in files:
    con = duckdb.connect()
    read_fn = get_read_fn(path)

    schema_df = con.execute(f"DESCRIBE SELECT * FROM {read_fn}").df()
    numeric_cols = [
        r['column_name'] for _, r in schema_df.iterrows()
        if any(t in r['column_type'].upper() for t in NUMERIC_TYPES)
    ]

    if not numeric_cols:
        print(f"{path.name}: No numeric columns detected")
        con.close()
        continue

    stat_exprs = ', '.join(
        f"MIN(\"{c}\") AS \"{c}__min\", MAX(\"{c}\") AS \"{c}__max\", "
        f"AVG(\"{c}\") AS \"{c}__avg\", STDDEV(\"{c}\") AS \"{c}__std\", "
        f"MEDIAN(\"{c}\") AS \"{c}__med\""
        for c in numeric_cols
    )
    stats_wide = con.execute(f"SELECT {stat_exprs} FROM {read_fn}").df()
    con.close()

    rows = []
    for c in numeric_cols:
        rows.append({
            'column': c,
            'min': stats_wide[f'{c}__min'].iloc[0],
            'max': stats_wide[f'{c}__max'].iloc[0],
            'mean': stats_wide[f'{c}__avg'].iloc[0],
            'std': stats_wide[f'{c}__std'].iloc[0],
            'median': stats_wide[f'{c}__med'].iloc[0],
        })
    stats_df = pd.DataFrame(rows).set_index('column').round(4)
    display(Markdown(f"### {path.name} \u2014 Numeric Distributions"))
    display(stats_df)

In [ ]:
for path in files:
    con = duckdb.connect()
    read_fn = get_read_fn(path)

    schema_df = con.execute(f"DESCRIBE SELECT * FROM {read_fn}").df()
    row_count = con.execute(f"SELECT COUNT(*) FROM {read_fn}").fetchone()[0]

    string_cols = [
        r['column_name'] for _, r in schema_df.iterrows()
        if 'VARCHAR' in r['column_type'].upper() or 'TEXT' in r['column_type'].upper()
    ]

    if not string_cols:
        print(f"{path.name}: No string/categorical columns detected")
        con.close()
        continue

    results = []
    for c in string_cols:
        uniq = con.execute(f"SELECT COUNT(DISTINCT \"{c}\") FROM {read_fn}").fetchone()[0]
        results.append({'column': c, 'unique_count': uniq, 'cardinality_pct': round(uniq / row_count * 100, 1)})

    display(Markdown(f"### {path.name} \u2014 Categorical Columns"))
    display(pd.DataFrame(results))

    # Top 10 values for low-cardinality columns (< 5% of rows)
    low_card = [r['column'] for r in results if r['cardinality_pct'] < 5.0]
    for c in low_card[:5]:
        top = con.execute(
            f"SELECT \"{c}\", COUNT(*) AS freq FROM {read_fn} "
            f"GROUP BY \"{c}\" ORDER BY freq DESC LIMIT 10"
        ).df()
        display(Markdown(f"**Top values \u2014 `{c}`:**"))
        display(top)

    con.close()

## 2. Load Data for Statistical Analysis

Loading dataset into pandas for distribution analysis, correlations, and missing-value visualization.

<!-- Decision 5: If dataset exceeds 500,000 rows, a 50,000-row random sample is used for all statistical computations. The DuckDB profiling section above always uses the full dataset. -->
**Sampling rule:** If dataset exceeds 500,000 rows, a 50,000-row random sample is used for all statistical computations. The profiling section above always uses the full dataset.

In [ ]:
SAMPLE_THRESHOLD = 500_000
dfs = {}

for path in files:
    con = duckdb.connect()
    read_fn = get_read_fn(path)

    row_count = con.execute(f"SELECT COUNT(*) FROM {read_fn}").fetchone()[0]

    if row_count > SAMPLE_THRESHOLD:
        df = con.execute(f"SELECT * FROM {read_fn} USING SAMPLE 50000").df()
        print(f"{path.name}: Loaded 50,000-row sample (full dataset: {row_count:,} rows)")
    else:
        df = con.execute(f"SELECT * FROM {read_fn}").df()
        print(f"{path.name}: Loaded full dataset ({row_count:,} rows \u00d7 {len(df.columns)} columns)")

    dfs[path.name] = df
    con.close()

## 3. Distribution Analysis

<!-- EDA-03 -->
Histograms, Q-Q plots, normality tests (Shapiro-Wilk for n≤5000, D'Agostino for n>5000), skewness, and kurtosis for all numeric features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for fname, df in dfs.items():
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print(f"{fname}: No numeric columns \u2014 skipping histograms")
        continue

    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3 * n_rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, col in enumerate(numeric_cols):
        axes_flat[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
        axes_flat[i].set_title(col, fontsize=9)
        axes_flat[i].set_ylabel('Frequency')

    for j in range(len(numeric_cols), len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f'{fname} \u2014 Numeric Distributions', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy import stats as sp_stats

for fname, df in dfs.items():
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        continue

    display(Markdown(f"### {fname} \u2014 Normality Tests"))

    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3 * n_rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    normality_results = []
    for i, col in enumerate(numeric_cols):
        series = df[col].dropna()
        sp_stats.probplot(series, dist='norm', plot=axes_flat[i])
        axes_flat[i].set_title(f'Q-Q: {col}', fontsize=9)

        n = len(series)
        if n <= 5000:
            stat, p = sp_stats.shapiro(series)
            test_name = 'Shapiro-Wilk'
        else:
            stat, p = sp_stats.normaltest(series)
            test_name = "D'Agostino"

        normality_results.append({
            'column': col, 'n': n, 'test': test_name,
            'statistic': round(stat, 4), 'p_value': round(p, 4),
            'is_normal (p>0.05)': p > 0.05,
            'skewness': round(float(series.skew()), 3),
            'kurtosis': round(float(series.kurtosis()), 3),
        })

    for j in range(len(numeric_cols), len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(normality_results).set_index('column'))

## 4. Correlation Analysis

<!-- EDA-04, Decision 6: method selected per feature type -->
Method selected per feature type:
- **Both numeric** → Pearson correlation
- **Both categorical** (object dtype or cardinality < 10% of rows) → Cramér's V
- **One numeric + one binary** → point-biserial

In [ ]:
from scipy.stats import chi2_contingency, pointbiserialr

def cramers_v(x, y):
    """Cram\u00e9r's V for two categorical Series."""
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    phi2 = chi2 / n
    r, k = ct.shape
    denom = min(k - 1, r - 1)
    return float(np.sqrt(phi2 / denom)) if denom > 0 else 0.0

def is_categorical(series, threshold=0.1):
    return (series.dtype == 'object' or series.dtype.name == 'category' or
            series.nunique() < max(10, len(series) * threshold))

def is_binary(series):
    return series.nunique() == 2

for fname, df in dfs.items():
    display(Markdown(f"### {fname} \u2014 Correlation Analysis"))

    # Pearson matrix for numeric columns
    num_df = df.select_dtypes(include='number')
    if len(num_df.columns) >= 2:
        corr_matrix = num_df.corr(method='pearson')
        display(Markdown("**Pearson Correlation (numeric features):**"))
        display(corr_matrix.round(3))

    # Mixed-type correlations
    cols = df.columns.tolist()
    mixed_results = []
    for i, c1 in enumerate(cols):
        for c2 in cols[i + 1:]:
            s1 = df[c1].dropna()
            s2 = df[c2].dropna()
            common = s1.index.intersection(s2.index)
            s1, s2 = s1[common], s2[common]
            if len(s1) < 10:
                continue

            cat1, cat2 = is_categorical(s1), is_categorical(s2)
            bin1, bin2 = is_binary(s1), is_binary(s2)

            if cat1 and cat2:
                v = cramers_v(s1, s2)
                mixed_results.append({'col1': c1, 'col2': c2, 'method': "Cram\u00e9r's V", 'value': round(v, 3)})
            elif not cat1 and bin2:
                r_val, p_val = pointbiserialr(s2.astype(float), s1.astype(float))
                mixed_results.append({'col1': c1, 'col2': c2, 'method': 'point-biserial', 'value': round(r_val, 3), 'p': round(p_val, 4)})
            elif bin1 and not cat2:
                r_val, p_val = pointbiserialr(s1.astype(float), s2.astype(float))
                mixed_results.append({'col1': c1, 'col2': c2, 'method': 'point-biserial', 'value': round(r_val, 3), 'p': round(p_val, 4)})

    if mixed_results:
        display(Markdown("**Mixed-type correlations:**"))
        display(pd.DataFrame(mixed_results))
    else:
        print("No mixed-type column pairs detected (or all numeric \u2014 see Pearson matrix above)")

In [ ]:
for fname, df in dfs.items():
    num_df = df.select_dtypes(include='number')
    if len(num_df.columns) < 2:
        print(f"{fname}: Need >= 2 numeric columns for heatmap")
        continue

    corr = num_df.corr(method='pearson')
    size = max(6, len(corr.columns) * 0.8)
    fig, ax = plt.subplots(figsize=(size, size * 0.9))
    sns.heatmap(
        corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
        vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
        annot_kws={'size': 8}
    )
    ax.set_title(f'{fname} \u2014 Pearson Correlation Heatmap', fontsize=11)
    plt.tight_layout()
    plt.show()

## 5. Missing Value Analysis

<!-- EDA-05, Decision 2 -->
Missingness heatmap, percentage by feature, and correlation of missingness.

In [ ]:
for fname, df in dfs.items():
    null_mask = df.isnull()
    total_missing = null_mask.values.sum()

    if total_missing == 0:
        print(f"{fname}: No missing values \u2014 skipping missingness plots")
        continue

    display(Markdown(f"### {fname} \u2014 Missing Value Analysis"))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(4, len(df.columns) * 0.5)))

    # Heatmap of null mask
    sns.heatmap(null_mask.T, cbar=False, yticklabels=True, xticklabels=False,
                cmap='viridis', ax=ax1)
    ax1.set_title('Missingness Heatmap\n(yellow = missing)')
    ax1.set_xlabel('Observations (rows)')

    # Percentage missing bar chart (sorted)
    pct = (null_mask.sum() / len(df) * 100).sort_values()
    pct_nonzero = pct[pct > 0]
    pct_nonzero.plot(kind='barh', ax=ax2, color='steelblue')
    ax2.set_title('% Missing by Feature')
    ax2.set_xlabel('% Missing')
    ax2.axvline(x=5, color='orange', linestyle='--', linewidth=1, label='5%')
    ax2.axvline(x=30, color='red', linestyle='--', linewidth=1, label='30%')
    ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f"Total missing values: {total_missing:,} ({total_missing / df.size * 100:.1f}% of all cells)")

In [ ]:
for fname, df in dfs.items():
    null_mask = df.isnull()
    missing_cols = null_mask.columns[null_mask.sum() > 0].tolist()

    if len(missing_cols) < 2:
        if len(missing_cols) == 1:
            print(f"{fname}: Only one column with missing values \u2014 no missingness correlation to show")
        continue

    miss_corr = null_mask[missing_cols].corr()
    size = max(5, len(missing_cols) * 0.9)
    fig, ax = plt.subplots(figsize=(size, size * 0.9))
    sns.heatmap(miss_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                vmin=-1, vmax=1, square=True, ax=ax, annot_kws={'size': 8})
    ax.set_title(f'{fname} \u2014 Missingness Correlation\n(correlated missingness \u2192 MAR or MNAR)')
    plt.tight_layout()
    plt.show()

### MCAR / MAR / MNAR Assessment

Use the heatmap and correlation matrix above to assess the missingness mechanism:

| Mechanism | What it means | Visual signal | Safe action |
|-----------|---------------|---------------|-------------|
| **MCAR** \u2014 Missing Completely At Random | Missingness unrelated to any variable | Heatmap shows random scatter; correlation matrix near zero | Complete-case analysis valid (drop rows with nulls) |
| **MAR** \u2014 Missing At Random | Missingness related to **observed** variables | Correlation matrix shows non-zero values between missing columns | Multiple imputation using correlated observed columns |
| **MNAR** \u2014 Missing Not At Random | Missingness related to the **missing value itself** | Cannot detect statistically \u2014 requires domain knowledge | Domain investigation; Heckman selection models |

**Analyst action:** Examine the missingness correlation matrix. Non-zero correlations between columns' missing indicators are evidence of MAR (at minimum) \u2014 check whether those columns are related in the data-generating process.

## 6. Time Series Analysis

<!-- EDA-06, Decision 6: activated when time_factor = true in config.json -->
*Activated when time series data is confirmed during project setup.*

Stationarity tests (ADF + KPSS) and seasonal decomposition for the first 1–2 numeric columns.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose

time_factor = config.get('time_factor', False)

if not time_factor:
    print("time_factor = False \u2014 time series analysis skipped.")
    print("To enable: set time_factor=true in .planning/config.json")
else:
    display(Markdown("### Time Series Stationarity & Decomposition"))

    for fname, df in dfs.items():
        numeric_cols = df.select_dtypes(include='number').columns.tolist()
        if not numeric_cols:
            print(f"{fname}: No numeric columns for time series tests")
            continue

        display(Markdown(f"#### {fname}"))

        for ts_col in numeric_cols[:2]:
            series = df[ts_col].dropna().reset_index(drop=True)
            if len(series) < 20:
                print(f"  {ts_col}: Too few observations (n={len(series)}) \u2014 skipping")
                continue

            display(Markdown(f"**`{ts_col}` (n={len(series)})**"))

            # ADF test (H0: unit root = non-stationary)
            try:
                adf = adfuller(series, autolag='AIC')
                print(f"ADF Test  \u2014 statistic: {adf[0]:.4f}, p-value: {adf[1]:.4f}, lags: {adf[2]}")
                for cv_k, cv_v in adf[4].items():
                    print(f"           critical value {cv_k}: {cv_v:.3f}")
                interp = 'STATIONARY (reject H0, p<0.05)' if adf[1] < 0.05 else 'NON-STATIONARY (fail to reject H0)'
                print(f"           \u2192 {interp}")
            except Exception as e:
                print(f"  ADF failed: {e}")

            # KPSS test (H0: stationary)
            try:
                kp = kpss(series, regression='c', nlags='auto')
                print(f"\nKPSS Test \u2014 statistic: {kp[0]:.4f}, p-value: {kp[1]:.4f}")
                interp = 'NON-STATIONARY (reject H0, p<0.05)' if kp[1] < 0.05 else 'STATIONARY (fail to reject H0)'
                print(f"           \u2192 {interp}")
            except Exception as e:
                print(f"  KPSS failed: {e}")

            # Seasonal decomposition (requires >= 2*period obs)
            try:
                period = 12
                if len(series) >= 2 * period:
                    decomp = seasonal_decompose(series, model='additive', period=period, extrapolate_trend='freq')
                    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
                    decomp.observed.plot(ax=axes[0], title='Observed', color='steelblue')
                    decomp.trend.plot(ax=axes[1], title='Trend', color='darkorange')
                    decomp.seasonal.plot(ax=axes[2], title='Seasonal', color='green')
                    decomp.resid.plot(ax=axes[3], title='Residual', color='gray')
                    fig.suptitle(f'{ts_col} \u2014 Seasonal Decomposition (period={period})', fontsize=11)
                    plt.tight_layout()
                    plt.show()
                else:
                    print(f"\n  Decomposition skipped: need >= {2*period} obs, have {len(series)}")
            except Exception as e:
                print(f"\n  Decomposition failed: {e}")
            print()

## 7. Tidy Data Validation

<!-- EDA-07, Decision 7: validate before feature engineering; flag violations, do not fix -->
Validates data against tidy principles **before** any feature engineering.

Checks:
1. Duplicate rows
2. Multiple values in one cell (comma/semicolon/pipe-separated strings)
3. Mixed observation types (column-name prefix heuristic)

Violations are **flagged only** — not silently fixed.

In [ ]:
import re

for fname, df in dfs.items():
    display(Markdown(f"### {fname} \u2014 Tidy Validation"))
    issues = []

    # Check 1: Duplicate rows
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        display(Markdown(f"**[VIOLATION] Duplicate rows: {dup_count} exact duplicates**"))
        display(df[df.duplicated(keep=False)].head(3))
        issues.append(f"Duplicate rows ({dup_count}) \u2014 action: `df.drop_duplicates()` before modelling")
    else:
        print("\u2713 No duplicate rows")

    # Check 2: Multiple values in one cell (>5% of non-null values contain ,;|)
    multi_cols = []
    for col in df.select_dtypes(include='object').columns:
        non_null = df[col].dropna()
        if len(non_null) == 0:
            continue
        pct = non_null.str.contains(r'[,;|]', regex=True, na=False).mean()
        if pct > 0.05:
            multi_cols.append({'column': col, 'pct_multi_value': round(pct * 100, 1)})
    if multi_cols:
        display(Markdown(f"**[VIOLATION] Multi-value cells (>5% contain comma/semicolon/pipe):**"))
        display(pd.DataFrame(multi_cols))
        issues.append(f"Multi-value columns: {[r['column'] for r in multi_cols]} \u2014 split into separate rows/columns")
    else:
        print("\u2713 No multi-value cells detected")

    # Check 3: Mixed observation types (column-name prefix heuristic)
    prefixes = {}
    for col in df.columns:
        prefix = col.split('_')[0].lower() if '_' in col else col[:4].lower()
        prefixes[prefix] = prefixes.get(prefix, 0) + 1
    dominant = {p for p, c in prefixes.items() if c >= 2}
    if len(dominant) >= 3:
        display(Markdown(
            f"**[POTENTIAL VIOLATION] Mixed entity types:** {len(dominant)} column-name prefixes suggest "
            f"multiple entity types \u2014 `{'`, `'.join(sorted(dominant)[:5])}`"
        ))
        issues.append("Mixed entity column prefixes \u2014 verify table does not combine multiple entities")
    else:
        print("\u2713 No obvious mixed entity types")

    if issues:
        display(Markdown("\n**Tidy violations summary \u2014 remediate before feature engineering:**"))
        for iss in issues:
            print(f"  \u2022 {iss}")
    else:
        print(f"\n\u2713 {fname}: All tidy checks passed")
    print()

## 8. Processed Dataset

<!-- EDA-08, Decision 4: structural cleaning only; write to data/processed/ -->
Structural cleaning only — writes output to `data/processed/`.

**Applied:**
- Parse unambiguous date string columns to datetime dtype

**Not applied (deferred to preprocessing):**
- Duplicate removal — duplicates are flagged in Section 7; dropping is a modelling decision
- Imputation — strategy depends on model; belongs inside `sklearn.Pipeline`
- Feature engineering — EDA identifies candidates; preprocessing implements them

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)  # safe: data/raw/ is read-only (INFR-05)

for path in files:
    con = duckdb.connect()
    read_fn = get_read_fn(path)
    df = con.execute(f"SELECT * FROM {read_fn}").df()
    con.close()

    # Parse unambiguous date string columns
    date_cols_parsed = []
    for col in df.select_dtypes(include='object').columns:
        sample = df[col].dropna().head(20)
        if len(sample) == 0:
            continue
        try:
            pd.to_datetime(sample, format='mixed', errors='raise')
            df = df.copy()
            df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')
            date_cols_parsed.append(col)
        except Exception:
            pass

    out_path = processed_dir / path.name
    ext = path.suffix.lower()
    if ext == '.csv':
        df.to_csv(out_path, index=False)
    elif ext == '.parquet':
        df.to_parquet(out_path, index=False)
    elif ext == '.xlsx':
        df.to_excel(out_path, index=False)

    print(f"{path.name}:")
    print(f"  Shape:   {len(df):,} rows \u00d7 {len(df.columns)} columns")
    if date_cols_parsed:
        print(f"  Parsed date columns: {date_cols_parsed}")
    print(f"  Written: {out_path}")
    print()

## Caveats

<!-- OUT-03: correlation is not causation disclaimer required in all stakeholder reports -->
- **Correlation is not causation** — all findings in this notebook are observational. Do not infer causal relationships without a controlled experiment.
- Distribution and correlation analysis was computed on the loaded dataset (see Data Loading section for sampling details if dataset exceeded 500,000 rows).
- Tidy violations are flagged but not remediated in this phase — review and fix before feature engineering.
- Imputation strategies are deferred to preprocessing, embedded in a pipeline appropriate to the chosen model.
- Time series stationarity tests are only activated when time series data was confirmed during project setup.
- The processed dataset in `data/processed/` reflects structural changes only. Raw data in `data/raw/` is unchanged.